<a href="https://colab.research.google.com/github/simmonsv19910816/Simmons_Victoria_ML_ITAI1371_12321_FINAL_EXAM/blob/main/app_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#download needed libraries,
%pip install kagglehub numpy pandas scikit-learn matplotlib


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# try to get the file or crash out if missing
try:
    df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')
    print("row count:", len(df))
except:
    print("fix path to Airbnb_Open_Data.csv")

# lower case cols and add underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# clean up the messy price columns
for c in ['price', 'service_fee']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df[c] = pd.to_numeric(df[c], errors='coerce')

print("\ncols ready:")
print(df[['price', 'service_fee']].dtypes)
display(df.head())

<>:7: SyntaxWarning: invalid escape sequence '\D'
<>:7: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_435/4289972517.py:7: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')


fix path to Airbnb_Open_Data.csv


NameError: name 'df' is not defined

In [ ]:
# nulls before any filling
nulls_before = df.isnull().sum()
nulls_before = nulls_before[nulls_before > 0].sort_values(ascending=False)
print("nulls before:\n", nulls_before)

# 1. fix neighborhood typos and blanks
if 'neighbourhood_group' in df.columns:
    df['neighbourhood_group'] = df['neighbourhood_group'].fillna('Unknown').replace({'brookln': 'Brooklyn', 'manhatan': 'Manhattan'})

# fill policy and room types with mode
df['cancellation_policy'] = df['cancellation_policy'].fillna(df['cancellation_policy'].mode()[0])
df['room_type'] = df['room_type'].fillna(df['room_type'].mode()[0])

# 2. replace all numerical nan types with their column median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 3. fill any leftover text columns with 'Unknown'
text_cols = df.select_dtypes(include=['object']).columns
df[text_cols] = df[text_cols].fillna('Unknown')

# nulls after filling (reindexed to match the before chart)
nulls_after = df.isnull().sum().reindex(nulls_before.index).fillna(0)

print("\nnulls left:")
print(df[['price', 'service_fee', 'construction_year', 'neighbourhood_group']].isnull().sum())

# before/after bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(nulls_before.index, nulls_before.values, color='tomato')
axes[0].set_title("null counts before")
axes[0].tick_params(axis='x', rotation=90)

axes[1].bar(nulls_before.index, nulls_after.values, color='seagreen')
axes[1].set_title("null counts after")
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

display(df.head())

In [ ]:
#remove rows with negative or zer0 values
rows_before = len(df)

bad_rows = df[df['price'] <= 0].index
df = df.drop(index=bad_rows).reset_index(drop=True)
rows_after = len(df)
print(f"dropped {len(bad_rows)} zero-price rows")

#fee ratio
df['fee_to_price_ratio'] = df['service_fee'] / df['price']
print(df['fee_to_price_ratio'].describe())

# before/after row count
plt.bar(['before', 'after'], [rows_before, rows_after], color=['tomato', 'seagreen'])
plt.title("row count before/after dropping zero-price rows")
plt.show()

# fee_to_price_ratio distribution (new engineered feature)
plt.hist(df['fee_to_price_ratio'], bins=50, color='steelblue')
plt.title("fee_to_price_ratio distribution")
plt.show()

display(df.head())

In [ ]:
# before plot - raw year range
plt.hist(df['construction_year'], bins=30, color='steelblue')
plt.title("construction_year before scaling")
plt.show()

# custom min max scaling for year built
y_min = df['construction_year'].min()
y_max = df['construction_year'].max()

if (y_max - y_min) != 0:
    df['construction_year_scaled'] = (df['construction_year'] - y_min) / (y_max - y_min)
else:
    df['construction_year_scaled'] = 0.0

print("scale check:")
print("min:", df['construction_year_scaled'].min(), "max:", df['construction_year_scaled'].max())

# after plot - same shape, just rescaled to [0,1]
plt.hist(df['construction_year_scaled'], bins=30, color='seagreen')
plt.title("construction_year_scaled after scaling")
plt.show()

display(df.head())

In [ ]:
# print original number_of_reviews skew
print("skew before:", df['number_of_reviews'].skew())

# before plot
plt.hist(df['number_of_reviews'], bins=50, color='steelblue')
plt.title("number_of_reviews before")
plt.show()

# log transform to fix the skew (this actually reshapes the distribution)
df['reviews_log'] = np.log1p(df['number_of_reviews'])

print("skew after:", df['reviews_log'].skew())

# after plot
plt.hist(df['reviews_log'], bins=50, color='seagreen')
plt.title("number_of_reviews after log transform")
plt.show()

display(df[['number_of_reviews', 'reviews_log']].head())

In [ ]:
# Use pd.get_dummies() targeting the room_type and cancellation_policy columns.
# Pass drop_first=True to eliminate redundant tracking data and prevent multi-collinearity (the dummy variable trap).
# Specify dtype=int to force the output to display as numbers (1/0) instead of Booleans (True/False).
df = pd.get_dummies(df, columns=['room_type', 'cancellation_policy'], drop_first=True, dtype=int)


In [ ]:
from sklearn.utils import resample

# Isolate the dataset into two distinct dataframes based on class membership (True vs False)
df_true = df[df['instant_bookable'] == True]
df_false = df[df['instant_bookable'] == False]

# Identify majority and minority dataframes
if len(df_true) < len(df_false):
    df_minority = df_true
    df_majority = df_false
else:
    df_minority = df_false
    df_majority = df_true

# Downsample majority class observations without replacement
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

# Recombine the dataframes using pd.concat() and invoke .reset_index(drop=True)
df = pd.concat([df_minority, df_majority_downsampled]).reset_index(drop=True)


In [ ]:
# Convert the neighbourhood_group column type to a categorical type using .astype('category')
df['neighbourhood_group'] = df['neighbourhood_group'].astype('category')

# Extract the underlying numerical mappings by calling the .cat.codes attribute, and write them to a new column named neighbourhood_group_code
df['neighbourhood_group_code'] = df['neighbourhood_group'].cat.codes

# Also convert host_identity_verified to category codes to make it numeric
if 'host_identity_verified' in df.columns:
    df['host_identity_verified'] = df['host_identity_verified'].astype('category').cat.codes


In [ ]:
import os

# Consolidate non-predictive metadata names into an exclusion list (id, name, host_id, host_name, etc.)
exclude_cols = ['id', 'name', 'host_id', 'host_name', 'country', 'country_code', 'license', 'house_rules', 'last_review', 'neighbourhood', 'neighbourhood_group']

# Drop the target features using df.drop(columns=...), filtering for features that are active in the current dataframe state
cols_to_drop = [c for c in exclude_cols if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Export the resulting dataframe via .to_csv() into the working folder as \DATA\Airbnb_Cleaned_Data.csv with index=False.
output_dir = os.path.join('.', 'DATA')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'Airbnb_Cleaned_Data.csv')
df.to_csv(output_path, index=False)


In [ ]:
from sklearn.model_selection import train_test_split


X = df.drop(columns=['instant_bookable'])
y = df['instant_bookable'].astype(int)


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)


X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_val shape: {y_val.shape}, y_test shape: {y_test.shape}")
print(X_train.nunique().sort_values())

In [ ]:
print(y.index.equals(X.index))
print(X_train.index.isin(y_train.index).all())
X_train.assign(target=y_train.values).corr()['target'].sort_values()

In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Define each mode
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=42)
}

# Fit every model on the training set
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained.")

In [ ]:


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Evaluate each trained model on the validation set
results = []
for name, model in trained_models.items():
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred)
    rec = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)
    auc = roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else None

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print("Validation set performance (sorted by F1-Score):")
display(results_df)

In [ ]:


from sklearn.ensemble import VotingClassifier
import numpy as np

# Pick the top 3
top3_names = results_df['Model'].head(3).tolist()
print("Top 3 models selected for the ensemble:", top3_names)
top3_estimators = [(name, trained_models[name]) for name in top3_names]


voting_clf = VotingClassifier(estimators=top3_estimators, voting='soft')
voting_clf.fit(X_train, y_train)

def evaluate(model_or_proba_fn, X, y, label, is_proba_fn=False):
    if is_proba_fn:
        proba = model_or_proba_fn(X)
        pred = (proba >= 0.5).astype(int)
    else:
        pred = model_or_proba_fn.predict(X)
        proba = model_or_proba_fn.predict_proba(X)[:, 1] if hasattr(model_or_proba_fn, "predict_proba") else None

    metrics = {
        'Accuracy': accuracy_score(y, pred),
        'Precision': precision_score(y, pred),
        'Recall': recall_score(y, pred),
        'F1-Score': f1_score(y, pred),
        'ROC-AUC': roc_auc_score(y, proba) if proba is not None else None
    }
    print(f"{label}: {metrics}")
    return metrics

print("\nVoting Classifier (soft voting of top 3):")
voting_val_metrics = evaluate(voting_clf, X_val, y_val, "Validation")
voting_test_metrics = evaluate(voting_clf, X_test, y_test, "Test")


val_weights = results_df.set_index('Model').loc[top3_names, 'Accuracy']
weights = (val_weights / val_weights.sum()).values

def bayesian_ensemble_proba(X):
    probs = np.array([trained_models[name].predict_proba(X)[:, 1] for name in top3_names])
    return np.average(probs, axis=0, weights=weights)

print("\nBayesian (accuracy-weighted) ensemble:")
bayes_val_metrics = evaluate(bayesian_ensemble_proba, X_val, y_val, "Validation", is_proba_fn=True)
bayes_test_metrics = evaluate(bayesian_ensemble_proba, X_test, y_test, "Test", is_proba_fn=True)


comparison_df = pd.DataFrame({
    'Voting - Val': voting_val_metrics,
    'Voting - Test': voting_test_metrics,
    'Bayesian - Val': bayes_val_metrics,
    'Bayesian - Test': bayes_test_metrics
}).T

print("\nEnsemble comparison:")
display(comparison_df)